# Hierarchical Topic Analysis

Fine-grained topic modeling and bottom-up hierarchy over all extracted questions.

In [1]:
# import sys
# import nbformat

# print(sys.executable)
# print(nbformat.__version__)

# %pip install --upgrade nbformat

In [2]:
import ast
import pickle as pkl
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from IPython.display import HTML, display
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import TSNE
from umap import UMAP

try:
    stopwords.words('portuguese')
except LookupError:
    nltk.download('stopwords')

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_DIR / 'data/perguntas/relevant_question_extraction_gpt-5-4-mini_high_flex.csv'
GOLD_PATH = PROJECT_DIR / 'data/files/queries.txt'
BASE_OUTPUT_DIR = PROJECT_DIR / 'results/hierarchical_topics'
selected_config = {'config_id': 'nn30_mcs20', 'n_neighbors': 30, 'n_components': 5, 'min_cluster_size': 20, 'min_samples': 15, 'cluster_selection_method': 'eom'}
RANDOM_STATE = 2024
OUTPUT_DIR = BASE_OUTPUT_DIR / 'nn30_mcs20_seed2024'
EMBEDDINGS_PATH = BASE_OUTPUT_DIR / 'embeddings_all_questions_google_embeddinggemma-300m.npy'
GOLD_EMBEDDINGS_PATH = BASE_OUTPUT_DIR / 'embeddings_gold_clustering.npy'
VISUAL_EMBEDDINGS_PATH = BASE_OUTPUT_DIR / 'embeddings_all_questions_sentence_similarity.npy'
GOLD_VISUAL_EMBEDDINGS_PATH = BASE_OUTPUT_DIR / 'embeddings_gold_sentence_similarity.npy'
SOURCE_EMBEDDINGS_PATH = PROJECT_DIR / 'results/topic_analysis/embeddings_google_embeddinggemma-300m.npy'
MODEL_PATH = OUTPUT_DIR / 'bertopic_fine_model.pkl'
EMBEDDING_MODEL = 'google/embeddinggemma-300m'
MIN_CLUSTER_SIZE = selected_config['min_cluster_size']
MIN_SAMPLES = selected_config['min_samples']
OUTLIER_STRATEGY = 'c-tf-idf'
OUTLIER_THRESHOLD = 0.1
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

/scratch/victoria.estanislau/g2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data and embeddings

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
df = df[df['perguntas'].ne('[]')].copy()
df['perguntas'] = df['perguntas'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df = df.explode('perguntas').dropna(subset=['perguntas']).reset_index(drop=True)
docs = df['perguntas'].astype(str).tolist()
unique_codes, unique_docs = pd.factorize(pd.Series(docs), sort=False)
unique_docs = unique_docs.tolist()
unique_first_indices = np.unique(unique_codes, return_index=True)[1]
unique_df = df.iloc[unique_first_indices].reset_index(drop=True).copy()
gold_questions = [line.strip() for line in GOLD_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
print(f'{len(docs):,} perguntas extraídas ({len(unique_docs):,} únicas)')
print(f'{len(gold_questions):,} perguntas padrão-ouro')

18,607 perguntas extraídas (16,165 únicas)
98 perguntas padrão-ouro


In [4]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embedding_model.max_seq_length = 512

if EMBEDDINGS_PATH.exists():
    embeddings = np.load(EMBEDDINGS_PATH)
elif SOURCE_EMBEDDINGS_PATH.exists():
    embeddings = np.load(SOURCE_EMBEDDINGS_PATH)
    np.save(EMBEDDINGS_PATH, embeddings)
else:
    embedding_docs = [f'task: clustering | query: {doc}' for doc in docs]
    embeddings = embedding_model.encode(embedding_docs, batch_size=8, show_progress_bar=True, normalize_embeddings=True)
    np.save(EMBEDDINGS_PATH, embeddings)

if GOLD_EMBEDDINGS_PATH.exists():
    gold_embeddings = np.load(GOLD_EMBEDDINGS_PATH)
else:
    gold_docs = [f'task: clustering | query: {question}' for question in gold_questions]
    gold_embeddings = embedding_model.encode(gold_docs, batch_size=8, show_progress_bar=True, normalize_embeddings=True)
    np.save(GOLD_EMBEDDINGS_PATH, gold_embeddings)

if VISUAL_EMBEDDINGS_PATH.exists():
    visual_embeddings = np.load(VISUAL_EMBEDDINGS_PATH)
else:
    visual_docs = [f'task: sentence similarity | query: {doc}' for doc in docs]
    visual_embeddings = embedding_model.encode(visual_docs, batch_size=8, show_progress_bar=True, normalize_embeddings=True)
    np.save(VISUAL_EMBEDDINGS_PATH, visual_embeddings)

if GOLD_VISUAL_EMBEDDINGS_PATH.exists():
    gold_visual_embeddings = np.load(GOLD_VISUAL_EMBEDDINGS_PATH)
else:
    gold_docs = [f'task: sentence similarity | query: {question}' for question in gold_questions]
    gold_visual_embeddings = embedding_model.encode(gold_docs, batch_size=8, show_progress_bar=True, normalize_embeddings=True)
    np.save(GOLD_VISUAL_EMBEDDINGS_PATH, gold_visual_embeddings)

assert len(embeddings) == len(docs)
assert len(gold_embeddings) == len(gold_questions)
assert len(visual_embeddings) == len(docs)
assert len(gold_visual_embeddings) == len(gold_questions)
unique_embeddings = embeddings[unique_first_indices]
unique_visual_embeddings = visual_embeddings[unique_first_indices]

Batches: 100%|██████████| 13/13 [00:00<00:00, 40.44it/s]


## Fine-grained topics

In [5]:
umap_model = UMAP(n_neighbors=selected_config['n_neighbors'], n_components=selected_config['n_components'], min_dist=0.0, metric='cosine', random_state=RANDOM_STATE)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, metric='euclidean', cluster_selection_method=selected_config['cluster_selection_method'], prediction_data=True)
vectorizer_model = CountVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, stop_words=list(stopwords.words('portuguese')))
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = MaximalMarginalRelevance(diversity=0.6)

model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    min_topic_size=MIN_CLUSTER_SIZE,
    top_n_words=15,
    verbose=True,
)
base_topics, probabilities = model.fit_transform(unique_docs, unique_embeddings)
base_topics = np.asarray(base_topics)
base_topic_names = model.get_topic_info().set_index('Topic')['Name']
fine_topics = np.asarray(model.reduce_outliers(
    unique_docs,
    base_topics.tolist(),
    strategy=OUTLIER_STRATEGY,
    threshold=OUTLIER_THRESHOLD,
))
model.update_topics(unique_docs, topics=fine_topics)

gold_topics_hdbscan, gold_probabilities = model.transform(gold_questions, gold_embeddings)
all_topic_ids = sorted(model.get_topics())
regular_topic_ids = [topic for topic in all_topic_ids if topic != -1]
regular_topic_embeddings = model.topic_embeddings_[[all_topic_ids.index(topic) for topic in regular_topic_ids]]
regular_topic_embeddings = regular_topic_embeddings / np.linalg.norm(regular_topic_embeddings, axis=1, keepdims=True)
gold_embeddings_normalized = gold_embeddings / np.linalg.norm(gold_embeddings, axis=1, keepdims=True)
gold_similarities = gold_embeddings_normalized @ regular_topic_embeddings.T
nearest_indices = gold_similarities.argmax(axis=1)
nearest_topics = np.asarray(regular_topic_ids)[nearest_indices]
nearest_scores = gold_similarities[np.arange(len(gold_questions)), nearest_indices]
gold_topics = np.asarray(gold_topics_hdbscan).copy()
fallback_mask = gold_topics == -1
gold_topics[fallback_mask] = nearest_topics[fallback_mask]
gold_assignment_method = np.where(fallback_mask, 'Topic embedding mais próximo', 'HDBSCAN transform')
if np.asarray(gold_probabilities).ndim == 2:
    gold_probability_scores = np.asarray(gold_probabilities).max(axis=1)
else:
    gold_probability_scores = np.asarray(gold_probabilities)
gold_assignment_score = np.where(fallback_mask, nearest_scores, gold_probability_scores)
assert set(gold_topics).issubset(set(regular_topic_ids))
with open(MODEL_PATH, 'wb') as file:
    pkl.dump(model, file)
print(f"Configuração: {selected_config['config_id']} | seed: {RANDOM_STATE} | reduce_outliers: {OUTLIER_STRATEGY}, threshold={OUTLIER_THRESHOLD}")

2026-09-12 19:54:00,437 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
/scratch/victoria.estanislau/g2/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)
2026-09-12 19:54:29,105 - BERTopic - Dimensionality - Completed ✓
2026-09-12 19:54:29,106 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-12 19:54:29,429 - BERTopic - Cluster - Completed ✓
2026-09-12 19:54:29,433 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-12 19:54:37,972 - BERTopic - Representation - Completed ✓
2026-09-12 19:54:38,311 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the la

Configuração: nn30_mcs20 | seed: 2024 | reduce_outliers: c-tf-idf, threshold=0.1


In [6]:
topic_info = model.get_topic_info()
topic_names = topic_info.set_index('Topic')['Name']
unique_df['source_topic_id'] = base_topics
unique_df['source_topic_name'] = unique_df['source_topic_id'].map(base_topic_names)
unique_df['fine_topic_id'] = fine_topics
unique_df['fine_topic_name'] = unique_df['fine_topic_id'].map(topic_names)
df['fine_topic_id'] = fine_topics[unique_codes]
df['fine_topic_name'] = df['fine_topic_id'].map(topic_names)
df.to_csv(OUTPUT_DIR / 'all_question_occurrences_with_topics.csv', index=False)
unique_df.to_csv(OUTPUT_DIR / 'unique_questions_with_fine_topics.csv', index=False)
topic_info.to_csv(OUTPUT_DIR / 'fine_topic_descriptions.csv', index=False)
gold_df = pd.DataFrame({'pergunta': gold_questions, 'hdbscan_topic_id': gold_topics_hdbscan, 'topic_id': gold_topics, 'assignment_method': gold_assignment_method, 'score': gold_assignment_score})
gold_df['topic_name'] = gold_df['topic_id'].map(topic_names).fillna('Ruído')
gold_df.to_csv(OUTPUT_DIR / 'gold_questions_with_topics.csv', index=False)

reassigned = ((base_topics == -1) & (fine_topics != -1)).sum()
remaining_outliers = (fine_topics == -1).sum()
clustered = (fine_topics != -1).mean() * 100
print(f'{(topic_info.Topic != -1).sum()} tópicos finos')
print(f'{clustered:.2f}% das perguntas clusterizadas')
print(f'{reassigned:,} outliers reatribuídos; {remaining_outliers:,} permaneceram em -1')
topic_info.head(10)

158 tópicos finos
97.35% das perguntas clusterizadas
4,090 outliers reatribuídos; 428 permaneceram em -1


,Topic,Count,Name,Representation,Representative_Docs
0,-1,428,-1_que_bebê_por_algo,"[que, bebê, por, algo, você, parto, caminho, c...",[O que pode significar um sangramento de um di...
1,0,1267,0_maternidade_salário_auxílio_direito,"[maternidade, salário, auxílio, direito, inss,...",[Posso solicitar o auxílio-maternidade depois ...
2,1,360,1_anticoncepcional_pílula_seguinte_tomar,"[anticoncepcional, pílula, seguinte, tomar, ca...",[Há risco de gravidez após esquecer dois compr...
3,2,361,2_progesterona_via_oral_vaginal,"[progesterona, via, oral, vaginal, uso, utroge...",[O uso diário de Utrogestan 100 mg pode causar...
4,3,308,3_ela_grávida_está_pessoa,"[ela, grávida, está, pessoa, mencionada, engra...",[A pessoa mencionada no comentário está grávid...
5,4,264,4_nome_for_nomes_vocês,"[nome, for, nomes, vocês, chamar, qual, menina...","[Qual seria o nome do bebê se fosse menino?, Q..."
6,5,195,5_nascer_fevereiro_dia_vai,"[nascer, fevereiro, dia, vai, janeiro, anivers...","[A bebê vai nascer no dia 30 de janeiro, que é..."
7,6,187,6_dilatação_semanas_38_contrações,"[dilatação, semanas, 38, contrações, tampão, m...",[É normal sentir muitas contrações de treiname...
8,7,182,7_podem_cólicas_seios_sinais,"[podem, cólicas, seios, sinais, dor, indicar, ...","[Mesmo tendo menstruado neste mês, cólicas lev..."
9,8,160,8_hcg_beta_mui_resultado,"[hcg, beta, mui, resultado, ml, quantitativo, ...",[Um resultado de beta-hCG de 25 mUI/mL indica ...


## Topic hierarchy

In [7]:
hierarchy_embeddings = model.hierarchical_topics(unique_docs, use_ctfidf=False)
hierarchy_ctfidf = model.hierarchical_topics(unique_docs, use_ctfidf=True)
hierarchy_embeddings.to_csv(OUTPUT_DIR / 'hierarchical_topics_embeddings.csv', index=False)
hierarchy_ctfidf.to_csv(OUTPUT_DIR / 'hierarchical_topics_ctfidf.csv', index=False)
embeddings_threshold = float(hierarchy_embeddings['Distance'].quantile(0.75))
ctfidf_threshold = float(hierarchy_ctfidf['Distance'].quantile(0.75))
hierarchy_embeddings_fig = model.visualize_hierarchy(hierarchical_topics=hierarchy_embeddings, use_ctfidf=False, color_threshold=embeddings_threshold)
hierarchy_ctfidf_fig = model.visualize_hierarchy(hierarchical_topics=hierarchy_ctfidf, use_ctfidf=True, color_threshold=ctfidf_threshold)
hierarchy_embeddings_fig.write_html(OUTPUT_DIR / 'topic_hierarchy_embeddings.html')
hierarchy_ctfidf_fig.write_html(OUTPUT_DIR / 'topic_hierarchy_ctfidf.html')
display(HTML('<h3>Topic embeddings</h3>' + hierarchy_embeddings_fig.to_html(full_html=False, include_plotlyjs='cdn')))
display(HTML('<h3>c-TF-IDF</h3>' + hierarchy_ctfidf_fig.to_html(full_html=False, include_plotlyjs='cdn')))

100%|██████████| 157/157 [00:00<00:00, 557.85it/s]


## Topic and hierarchy title lists

In [8]:
cluster_titles = topic_info.loc[topic_info['Topic'] != -1, ['Topic', 'Count']].rename(columns={'Topic': 'Topic_ID'})
cluster_titles['Title'] = cluster_titles['Topic_ID'].apply(lambda topic: ' | '.join(word for word, _ in model.get_topic(topic)[:5]))
cluster_titles = cluster_titles[['Topic_ID', 'Title', 'Count']].sort_values('Topic_ID').reset_index(drop=True)

def hierarchy_title_table(hierarchy, method):
    table = hierarchy[['Parent_ID', 'Parent_Name', 'Topics', 'Child_Left_ID', 'Child_Left_Name', 'Child_Right_ID', 'Child_Right_Name', 'Distance']].copy()
    table.insert(0, 'Method', method)
    table['Topic_Count'] = table['Topics'].apply(len)
    for column in ['Parent_Name', 'Child_Left_Name', 'Child_Right_Name']:
        table[column] = table[column].str.replace('_', ' | ', regex=False)
    return table

hierarchy_titles_embeddings = hierarchy_title_table(hierarchy_embeddings, 'Topic embeddings')
hierarchy_titles_ctfidf = hierarchy_title_table(hierarchy_ctfidf, 'c-TF-IDF')
hierarchy_titles_comparison = pd.concat([hierarchy_titles_embeddings, hierarchy_titles_ctfidf], ignore_index=True)
cluster_titles.to_csv(OUTPUT_DIR / 'cluster_titles.csv', index=False)
hierarchy_titles_embeddings.to_csv(OUTPUT_DIR / 'hierarchy_titles_embeddings.csv', index=False)
hierarchy_titles_ctfidf.to_csv(OUTPUT_DIR / 'hierarchy_titles_ctfidf.csv', index=False)
hierarchy_titles_comparison.to_csv(OUTPUT_DIR / 'hierarchy_titles_comparison.csv', index=False)
display(cluster_titles)
display(hierarchy_titles_comparison)

,Topic_ID,Title,Count
0,0,maternidade | salário | auxílio | direito | inss,1267
1,1,anticoncepcional | pílula | seguinte | tomar |...,360
2,2,progesterona | via | oral | vaginal | uso,361
3,3,ela | grávida | está | pessoa | mencionada,308
4,4,nome | for | nomes | vocês | chamar,264
...,...,...,...
153,153,cabelo | crescer | materno | crescimento | queda,63
154,154,teste | devo | fazer | atrasar | repetir,61
155,155,reflexo | vermelho | olho | preto | aparecer,42
156,156,ferro | suplementação | ferritina | endovenoso...,40


,Method,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance,Topic_Count
0,Topic embeddings,314,de | que | com | bebê | para,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",312,gravidez | de | maternidade | com | parto,313,leite | bebê | vai | pai | olívia,0.764085,158
1,Topic embeddings,313,leite | bebê | vai | pai | olívia,"[4, 10, 11, 16, 17, 21, 28, 29, 32, 34, 37, 38...",309,vai | olívia | pai | nome | grávida,306,leite | materno | bebê | mamadeira | vacina,0.584503,52
2,Topic embeddings,312,gravidez | de | maternidade | com | parto,"[0, 1, 2, 3, 5, 6, 7, 8, 9, 12, 13, 14, 15, 18...",307,menstruação | progesterona | teste | anticonce...,311,maternidade | parto | semanas | salário | gest...,0.556385,106
3,Topic embeddings,311,maternidade | parto | semanas | salário | gest...,"[0, 3, 5, 6, 8, 12, 13, 15, 19, 20, 22, 23, 24...",302,semanas | gravidez | gestação | hcg | beta,310,maternidade | salário | parto | auxílio | cesárea,0.396390,75
4,Topic embeddings,310,maternidade | salário | parto | auxílio | cesárea,"[0, 3, 5, 13, 15, 19, 20, 22, 23, 24, 25, 27, ...",308,parto | cesárea | bebê | que | normal,290,maternidade | salário | auxílio | direito | re...,0.370639,49
...,...,...,...,...,...,...,...,...,...,...
309,c-TF-IDF,162,pai | quem | homem | onde | criança,"[55, 112]",112,pai | quem | estiver | mencionada | nessa,55,pai | homem | quem | onde | filho,0.328191,2
310,c-TF-IDF,161,nascer | fevereiro | vai | dia | janeiro,"[5, 28, 79]",5,nascer | fevereiro | dia | vai | janeiro,158,olívia | nascer | vai | dia | fevereiro,0.306905,3
311,c-TF-IDF,160,falsa | barriga | mostrada | mentira | parece,"[48, 65]",48,falsa | barriga | mostrada | parece | mentira,65,barriga | falsa | mostrada | mentira | tamanho,0.211108,2
312,c-TF-IDF,159,menina | menino | será | ou | bebê,"[33, 132]",33,menina | menino | será | tabela | bebê,132,menino | menina | acha | certeza | será,0.190783,2


In [9]:
print(model.get_topic_tree(hierarchy_embeddings))

.
├─gravidez_de_maternidade_com_parto
│    ├─menstruação_progesterona_teste_anticoncepcional_atraso
│    │    ├─progesterona_via_útero_vaginal_histerectomia
│    │    │    ├─■──mounjaro_enquanto_amamentando_alguma_amamentava ── Topic: 119
│    │    │    └─progesterona_via_útero_vaginal_histerectomia
│    │    │         ├─progesterona_via_oral_vaginal_reposição
│    │    │         │    ├─■──progesterona_via_oral_vaginal_uso ── Topic: 2
│    │    │         │    └─útero_progesterona_ovários_histerectomia_usar
│    │    │         │         ├─■──útero_ovários_progesterona_usar_tem ── Topic: 130
│    │    │         │         └─■──histerectomia_total_progesterona_fez_repor ── Topic: 124
│    │    │         └─histerectomia_mioma_miomas_cirurgia_retirada
│    │    │              ├─■──histerectomia_retirada_cirurgia_útero_retirar ── Topic: 73
│    │    │              └─■──mioma_miomas_42_cirurgia_cm ── Topic: 89
│    │    └─menstruação_teste_atraso_anticoncepcional_menstrual
│    │         ├─ant

## Interactive cluster map

In [10]:
source_mask = unique_df['fine_topic_id'] != -1
source_plot_df = unique_df.loc[source_mask, ['perguntas', 'source_topic_id', 'source_topic_name', 'fine_topic_id', 'fine_topic_name']].copy()
plot_embeddings = unique_visual_embeddings[source_mask.to_numpy()]
plot_embeddings = np.vstack([plot_embeddings, gold_visual_embeddings])
plot_embeddings = plot_embeddings / np.linalg.norm(plot_embeddings, axis=1, keepdims=True)
points_50d = PCA(n_components=50).fit_transform(plot_embeddings)
points_2d = TSNE(n_components=2, random_state=RANDOM_STATE, init='pca').fit_transform(points_50d)
question_points = points_2d[:len(source_plot_df)]
gold_points = points_2d[len(source_plot_df):]
source_plot_df['x'] = question_points[:, 0]
source_plot_df['y'] = question_points[:, 1]
plot_df = source_plot_df[source_plot_df['fine_topic_id'] != -1].copy()
plot_df['cluster'] = plot_df['fine_topic_id'].astype(str)
plot_df['cluster_size'] = plot_df.groupby('fine_topic_id')['fine_topic_id'].transform('size')
gold_customdata = gold_df[['topic_id', 'topic_name', 'assignment_method', 'score']].to_numpy()

In [11]:
fig = px.scatter(
    plot_df,
    x='x',
    y='y',
    color='cluster',
    hover_name='perguntas',
    hover_data={'x': False, 'y': False, 'fine_topic_name': True, 'cluster_size': True},
    render_mode='webgl',
    title=f"{selected_config['config_id']} seed {RANDOM_STATE} — perguntas únicas por tópico (sem ruído)",
)
fig.update_traces(marker={'size': 4, 'opacity': 0.65})
fig.add_trace(go.Scattergl(
    x=gold_points[:, 0],
    y=gold_points[:, 1],
    mode='markers',
    name='Padrão-ouro',
    text=gold_questions,
    customdata=gold_customdata,
    marker={'color': '#d62728', 'size': 10, 'symbol': 'x'},
    hovertemplate='<b>%{text}</b><br>Cluster: %{customdata[0]}<br>Tópico: %{customdata[1]}<br>Método: %{customdata[2]}<br>Score: %{customdata[3]:.3f}<extra>Padrão-ouro</extra>',
))
fig.update_layout(width=1100, height=750, legend_title='Cluster', template='plotly_white')
fig.write_html(OUTPUT_DIR / 'fine_topic_map.html')
display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

## Interactive cluster focus

In [12]:
from IPython.display import FileLink

focus_df = source_plot_df.copy()
focus_df['status'] = np.where(focus_df['source_topic_id'] == -1, 'Reatribuída por c-TF-IDF', 'Original')
focus_df['cluster'] = focus_df['fine_topic_id'].astype(str)
focus_df['cluster_key'] = 'final:' + focus_df['fine_topic_id'].astype(str)
focus_df['topic_name'] = focus_df['fine_topic_name']
focus_df['cluster_size'] = focus_df.groupby('cluster_key')['cluster_key'].transform('size')
focus_customdata = focus_df[['cluster_key', 'cluster', 'topic_name', 'cluster_size', 'status', 'fine_topic_id']].to_numpy()
focus_fig = go.Figure()
focus_fig.add_trace(go.Scattergl(
    x=focus_df['x'],
    y=focus_df['y'],
    mode='markers',
    name='Perguntas',
    text=focus_df['perguntas'],
    customdata=focus_customdata,
    marker={'color': '#9ca3af', 'size': 4, 'opacity': 0.45},
    selected={'marker': {'color': '#2563eb', 'size': 6, 'opacity': 0.95}},
    unselected={'marker': {'color': '#e5e7eb', 'size': 4, 'opacity': 0.55}},
    hoverlabel={'bgcolor': '#6b7280', 'font': {'color': 'white'}},
    hovertemplate='<b>%{text}</b><br>Cluster: %{customdata[1]}<br>Tópico: %{customdata[2]}<br>Tamanho: %{customdata[3]}<br>Status: %{customdata[4]}<extra></extra>',
))
focus_fig.add_trace(go.Scattergl(
    x=gold_points[:, 0],
    y=gold_points[:, 1],
    mode='markers',
    name='Padrão-ouro',
    text=gold_questions,
    customdata=gold_customdata,
    marker={'color': '#d62728', 'size': 10, 'symbol': 'x', 'opacity': 0.9},
    hoverlabel={'bgcolor': '#d62728', 'font': {'color': 'white'}},
    hovertemplate='<b>%{text}</b><br>Cluster: %{customdata[0]}<br>Tópico: %{customdata[1]}<br>Método: %{customdata[2]}<br>Score: %{customdata[3]:.3f}<extra>Padrão-ouro</extra>',
))
focus_fig.update_layout(
    width=1100,
    height=750,
    template='plotly_white',
    title=f"{selected_config['config_id']} seed {RANDOM_STATE} — clique para destacar a fragmentação",
    showlegend=True,
)
focus_script = """
const plot = document.getElementById('{plot_id}');
const questionTrace = plot.data.findIndex(trace => trace.name === 'Perguntas');
const goldTrace = plot.data.findIndex(trace => trace.name === 'Padrão-ouro');
const sourceClusters = {};
const finalClusters = {};
plot.data[questionTrace].customdata.forEach((row, i) => {
    const sourceCluster = String(row[0]);
    const finalCluster = String(row[5]);
    if (!sourceClusters[sourceCluster]) sourceClusters[sourceCluster] = [];
    if (!finalClusters[finalCluster]) finalClusters[finalCluster] = [];
    sourceClusters[sourceCluster].push(i);
    finalClusters[finalCluster].push(i);
});
let lockedIndices = null;
const highlight = indices => Plotly.restyle(plot, {'selectedpoints': [indices]}, [questionTrace]);
const restore = () => Plotly.restyle(plot, {'selectedpoints': [null]}, [questionTrace]);
const colorHoverLabel = color => requestAnimationFrame(() => {
    plot.querySelectorAll('.hoverlayer .hovertext path').forEach(path => {
        path.style.fill = color;
        path.style.stroke = color;
    });
});
plot.on('plotly_hover', data => {
    if (data.points[0].curveNumber !== questionTrace) return;
    const cluster = String(data.points[0].customdata[0]);
    const index = data.points[0].pointIndex;
    colorHoverLabel(lockedIndices === null || lockedIndices.includes(index) ? '#2563eb' : '#6b7280');
    if (lockedIndices === null) highlight(sourceClusters[cluster]);
});
plot.on('plotly_unhover', () => {
    if (lockedIndices === null) restore();
});
plot.on('plotly_click', data => {
    const point = data.points[0];
    if (point.curveNumber === goldTrace) {
        const cluster = String(point.customdata[0]);
        lockedIndices = finalClusters[cluster] || [];
        highlight(lockedIndices);
    } else if (point.curveNumber === questionTrace && lockedIndices !== null) {
        lockedIndices = null;
        restore();
    } else if (point.curveNumber === questionTrace) {
        const cluster = String(point.customdata[0]);
        lockedIndices = sourceClusters[cluster];
        highlight(lockedIndices);
    }
});
plot.on('plotly_doubleclick', () => {
    lockedIndices = null;
    restore();
    return false;
});
"""
focus_path = OUTPUT_DIR / 'cluster_focus_map.html'
focus_fig.write_html(focus_path, include_plotlyjs='directory', post_script=focus_script)
display(FileLink(focus_path))

/scratch/victoria.estanislau/projeto-gravidez/results/hierarchical_topics/nn30_mcs20_seed2024/cluster_focus_map.html